In [1]:
# Standard library
import os
from pathlib import Path

# Data manipulation
import pandas as pd
import numpy as np

# Machine learning utilities
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler

# Visualization (para validar o split visualmente)
import matplotlib.pyplot as plt
import seaborn as sns

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Paths (relativos à raiz do projeto)
PROJECT_ROOT = Path.cwd().parent  # assume que corremos a partir de /notebooks
DATA_DIR = PROJECT_ROOT / "data"
CACHE_DIR = DATA_DIR / "cache"
CACHE_DIR.mkdir(exist_ok=True)  # cria a pasta se não existir

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory exists: {DATA_DIR.exists()}")

Project root: /Users/rennandamiani/Documents/ISAG/ProjetoFinal_PetFinder
Data directory exists: True


In [2]:
# Load the raw training data
train_csv_path = DATA_DIR / "train" / "train.csv"
df = pd.read_csv(train_csv_path)

# Basic sanity checks
print(f"Shape: {df.shape}")
print(f"Expected: (14993, 24)")
print()
print(f"Missing values per column (only columns with NaNs):")
print(df.isna().sum()[df.isna().sum() > 0])
print()
print(f"First 3 rows:")
df.head(3)

Shape: (14993, 24)
Expected: (14993, 24)

Missing values per column (only columns with NaNs):
Name           1265
Description      13
dtype: int64

First 3 rows:


,Type,Name,Age,Breed1,Breed2,Gender,Color1,Color2,Color3,MaturitySize,FurLength,Vaccinated,Dewormed,Sterilized,Health,Quantity,Fee,State,RescuerID,VideoAmt,Description,PetID,PhotoAmt,AdoptionSpeed
0,2,Nibble,3,299,0,1,1,7,0,1,1,2,2,2,1,1,100,41326,8480853f516546f6cf33aa88cd76c379,0,Nibble is a 3+ month old ball of cuteness. He ...,86e1089a3,1.0,2
1,2,No Name Yet,1,265,0,1,1,2,0,2,2,3,3,3,1,1,0,41401,3082c7125d8fb66f7dd4bff4192c8b14,0,I just found it alone yesterday near my apartm...,6296e909a,2.0,0
2,1,Brisco,1,307,0,1,2,7,0,2,2,1,1,2,1,1,0,41326,fa90fa5b1ee11c86938398b60abc32cb,0,Their pregnant mother was dumped by her irresp...,3422e4906,7.0,3


In [3]:
# Começamos a partir de uma cópia — nunca modificamos o df original
df_clean = df.copy()

# ============================================================
# 1. has_photo — sinal univariado mais forte da EDA (-22.7pp)
# ============================================================
df_clean['has_photo'] = (df_clean['PhotoAmt'] > 0).astype(int)

# ============================================================
# 2. is_group — ninhadas adotam ligeiramente mais devagar (-4.5pp)
# ============================================================
df_clean['is_group'] = (df_clean['Quantity'] > 1).astype(int)

# ============================================================
# 3. is_mixed_breed — simplificação do Breed1 (alta cardinalidade)
# ============================================================
# IDs 307 (Mixed Breed) e 264 (Domestic Short Hair) = ~40% do dataset
MIXED_BREED_IDS = {307, 264}
df_clean['is_mixed_breed'] = df_clean['Breed1'].isin(MIXED_BREED_IDS).astype(int)

# ============================================================
# 4. has_fee — o valor contínuo de Fee não discrimina bem
# ============================================================
df_clean['has_fee'] = (df_clean['Fee'] > 0).astype(int)

# ============================================================
# 5. Corrigir as 12 inconsistências Type × Breed1
# ============================================================
# Todos os 12 casos: declarados como Gato (Type=2) mas Breed1 é raça de cão
# Regra: confiar no Breed1, corrigir o Type

# Carregar o dicionário de raças para detetar inconsistências
breed_labels = pd.read_csv(DATA_DIR / "breed_labels.csv")
dog_breed_ids = set(breed_labels.loc[breed_labels['Type'] == 1, 'BreedID'])
cat_breed_ids = set(breed_labels.loc[breed_labels['Type'] == 2, 'BreedID'])

# Marcar linhas onde o Type declarado não bate com o tipo real do Breed1
inconsistent_mask = (
    ((df_clean['Type'] == 2) & (df_clean['Breed1'].isin(dog_breed_ids))) |
    ((df_clean['Type'] == 1) & (df_clean['Breed1'].isin(cat_breed_ids)))
)
n_inconsistent = inconsistent_mask.sum()
print(f"Inconsistências Type × Breed1 detetadas: {n_inconsistent}")

# Correção: confiar no Breed1, ajustar o Type em conformidade
df_clean.loc[inconsistent_mask & df_clean['Breed1'].isin(dog_breed_ids), 'Type'] = 1
df_clean.loc[inconsistent_mask & df_clean['Breed1'].isin(cat_breed_ids), 'Type'] = 2

# ============================================================
# 6. Target binário: AdoptionFast (conforme CLAUDE.md)
# ============================================================
df_clean['AdoptionFast'] = (df_clean['AdoptionSpeed'] <= 2).astype(int)

# ============================================================
# Verificação — distribuições das novas features
# ============================================================
print("\nNovas features — contagem de valores:")
for col in ['has_photo', 'is_group', 'is_mixed_breed', 'has_fee', 'AdoptionFast']:
    counts = df_clean[col].value_counts().sort_index()
    pct = df_clean[col].value_counts(normalize=True).sort_index() * 100
    print(f"\n{col}:")
    for val, count in counts.items():
        print(f"  {val}: {count:>6} ({pct[val]:.2f}%)")

Inconsistências Type × Breed1 detetadas: 12

Novas features — contagem de valores:

has_photo:
  0:    341 (2.27%)
  1:  14652 (97.73%)

is_group:
  0:  11565 (77.14%)
  1:   3428 (22.86%)

is_mixed_breed:
  0:   8770 (58.49%)
  1:   6223 (41.51%)

has_fee:
  0:  12663 (84.46%)
  1:   2330 (15.54%)

AdoptionFast:
  0:   7456 (49.73%)
  1:   7537 (50.27%)


In [4]:
# ============================================================
# 1. Age — aplicar log1p para comprimir a cauda longa
# ============================================================
# Age vai de 0 a 255 meses, com distribuição muito enviesada
# log1p = log(1 + x), que funciona mesmo para Age=0 (evita log(0) = -inf)
df_clean['Age_log'] = np.log1p(df_clean['Age'])

print("Age — antes e depois do log1p:")
print(df_clean[['Age', 'Age_log']].describe().round(2))

# ============================================================
# 2. VideoAmt — eliminar (sinal desprezável na EDA)
# ============================================================
# 95%+ dos animais têm VideoAmt=0, não vale o ruído
df_clean = df_clean.drop(columns=['VideoAmt'])
print(f"\nColuna VideoAmt removida. Shape atual: {df_clean.shape}")

# ============================================================
# 3. State — agrupar em 4 buckets (top-3 + "Outros")
# ============================================================
# O State bruto tem 15 valores únicos, a maioria com pouquíssimas amostras
top_states = df_clean['State'].value_counts().head(3).index.tolist()
print(f"\nTop-3 States (mantidos individualmente): {top_states}")

# Criar nova coluna State_grouped: mantém os top-3, agrupa o resto como "Other"
df_clean['State_grouped'] = df_clean['State'].where(
    df_clean['State'].isin(top_states),
    other=-1  # usamos -1 como código para "Outros" (nenhum StateID real é -1)
)

print("\nDistribuição de State_grouped:")
print(df_clean['State_grouped'].value_counts().sort_index())

# ============================================================
# 4. Verificação — colunas atuais do df_clean
# ============================================================
print(f"\nShape final: {df_clean.shape}")
print(f"\nColunas disponíveis:")
for col in df_clean.columns:
    print(f"  - {col}")

Age — antes e depois do log1p:
            Age   Age_log
count  14993.00  14993.00
mean      10.45      1.78
std       18.16      1.02
min        0.00      0.00
25%        2.00      1.10
50%        3.00      1.39
75%       12.00      2.56
max      255.00      5.55

Coluna VideoAmt removida. Shape atual: (14993, 29)

Top-3 States (mantidos individualmente): [41326, 41401, 41327]

Distribuição de State_grouped:
State_grouped
-1        1591
 41326    8714
 41327     843
 41401    3845
Name: count, dtype: int64

Shape final: (14993, 30)

Colunas disponíveis:
  - Type
  - Name
  - Age
  - Breed1
  - Breed2
  - Gender
  - Color1
  - Color2
  - Color3
  - MaturitySize
  - FurLength
  - Vaccinated
  - Dewormed
  - Sterilized
  - Health
  - Quantity
  - Fee
  - State
  - RescuerID
  - Description
  - PetID
  - PhotoAmt
  - AdoptionSpeed
  - has_photo
  - is_group
  - is_mixed_breed
  - has_fee
  - AdoptionFast
  - Age_log
  - State_grouped


In [5]:
# ============================================================
# Objetivo: criar 3 conjuntos (train/val/test) sem overlap de RescuerID
# ============================================================

# Preparar os arrays de entrada
X = df_clean.drop(columns=['AdoptionFast', 'AdoptionSpeed'])  # features
y = df_clean['AdoptionFast'].values                            # target
groups = df_clean['RescuerID'].values                          # grupos para não misturar

print(f"Total de linhas: {len(X)}")
print(f"RescuerIDs únicos: {df_clean['RescuerID'].nunique()}")
print(f"Distribuição do target: {np.bincount(y)} ({y.mean()*100:.2f}% positivos)")

# ============================================================
# Passo 1: separar TEST (~20%) usando StratifiedGroupKFold com 5 folds
# ============================================================
# Usamos 5 folds → cada fold tem ~20%. Pegamos no fold 0 como test.
sgkf_test = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

# next() avança o iterador e devolve o primeiro split
trainval_idx, test_idx = next(sgkf_test.split(X, y, groups))

X_trainval = X.iloc[trainval_idx].copy()
X_test = X.iloc[test_idx].copy()
y_trainval = y[trainval_idx]
y_test = y[test_idx]
groups_trainval = groups[trainval_idx]

print(f"\nApós separação de test:")
print(f"  Train+Val: {len(X_trainval)} linhas")
print(f"  Test:      {len(X_test)} linhas ({len(X_test)/len(X)*100:.1f}%)")

# ============================================================
# Passo 2: separar TRAIN (~80% de trainval) e VAL (~20% de trainval)
# ============================================================
# Dentro do trainval, aplicamos novamente 5-fold e pegamos no fold 0
sgkf_val = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

train_idx, val_idx = next(sgkf_val.split(X_trainval, y_trainval, groups_trainval))

X_train = X_trainval.iloc[train_idx].copy()
X_val = X_trainval.iloc[val_idx].copy()
y_train = y_trainval[train_idx]
y_val = y_trainval[val_idx]

print(f"\nApós separação de train/val:")
print(f"  Train: {len(X_train)} linhas ({len(X_train)/len(X)*100:.1f}%)")
print(f"  Val:   {len(X_val)} linhas ({len(X_val)/len(X)*100:.1f}%)")
print(f"  Test:  {len(X_test)} linhas ({len(X_test)/len(X)*100:.1f}%)")

# ============================================================
# Verificação 1: balanceamento do target em cada split
# ============================================================
print("\n--- Distribuição do target (AdoptionFast=1) ---")
print(f"Train: {y_train.mean()*100:.2f}%")
print(f"Val:   {y_val.mean()*100:.2f}%")
print(f"Test:  {y_test.mean()*100:.2f}%")
print(f"(Esperado: ~50.27%)")

# ============================================================
# Verificação 2: ZERO overlap de RescuerID entre splits
# ============================================================
rescuers_train = set(X_train['RescuerID'])
rescuers_val = set(X_val['RescuerID'])
rescuers_test = set(X_test['RescuerID'])

overlap_train_val = rescuers_train & rescuers_val
overlap_train_test = rescuers_train & rescuers_test
overlap_val_test = rescuers_val & rescuers_test

print("\n--- Verificação de overlap de RescuerID ---")
print(f"Train ∩ Val:  {len(overlap_train_val)} rescuers (esperado: 0)")
print(f"Train ∩ Test: {len(overlap_train_test)} rescuers (esperado: 0)")
print(f"Val ∩ Test:   {len(overlap_val_test)} rescuers (esperado: 0)")

# Assert explícito — se houver overlap, a célula falha aqui
assert len(overlap_train_val) == 0, "LEAK: RescuerID partilhado entre Train e Val!"
assert len(overlap_train_test) == 0, "LEAK: RescuerID partilhado entre Train e Test!"
assert len(overlap_val_test) == 0, "LEAK: RescuerID partilhado entre Val e Test!"
print("\n✅ Zero overlap confirmado. Split é seguro.")

Total de linhas: 14993
RescuerIDs únicos: 5595
Distribuição do target: [7456 7537] (50.27% positivos)

Após separação de test:
  Train+Val: 11996 linhas
  Test:      2997 linhas (20.0%)

Após separação de train/val:
  Train: 9600 linhas (64.0%)
  Val:   2396 linhas (16.0%)
  Test:  2997 linhas (20.0%)

--- Distribuição do target (AdoptionFast=1) ---
Train: 50.27%
Val:   50.29%
Test:  50.25%
(Esperado: ~50.27%)

--- Verificação de overlap de RescuerID ---
Train ∩ Val:  0 rescuers (esperado: 0)
Train ∩ Test: 0 rescuers (esperado: 0)
Val ∩ Test:   0 rescuers (esperado: 0)

✅ Zero overlap confirmado. Split é seguro.


In [6]:
# ============================================================
# Identificar top-20 Breed1 APENAS a partir do train
# ============================================================
# Regra de ouro: qualquer decisão tomada com base nos dados
# tem de ser tomada SÓ com o train, nunca com val/test.
top20_breeds = X_train['Breed1'].value_counts().head(20).index.tolist()
print(f"Top-20 Breed1 identificadas no train: {len(top20_breeds)} raças")
print(f"Cobrem {X_train['Breed1'].isin(top20_breeds).mean()*100:.1f}% do train")

# Criar uma coluna Breed1_grouped: mantém as top-20, restantes viram -1
def group_breeds(df, top_breeds):
    df = df.copy()
    df['Breed1_grouped'] = df['Breed1'].where(df['Breed1'].isin(top_breeds), other=-1)
    return df

X_train = group_breeds(X_train, top20_breeds)
X_val = group_breeds(X_val, top20_breeds)
X_test = group_breeds(X_test, top20_breeds)

# ============================================================
# One-hot encoding de Color1, State_grouped e Breed1_grouped
# ============================================================
# Usamos pd.get_dummies com uma lista fixa de categorias para
# garantir que train/val/test ficam com as MESMAS colunas.
categorical_cols = ['Color1', 'State_grouped', 'Breed1_grouped']

# Referência de categorias: as que existem no TRAIN
# (se uma categoria só aparece em val/test, não vale a pena modelar)
category_reference = {
    col: sorted(X_train[col].unique().tolist())
    for col in categorical_cols
}

print("\nCategorias por coluna (do train):")
for col, cats in category_reference.items():
    print(f"  {col}: {len(cats)} valores")

# Função que aplica o one-hot encoding de forma consistente
def apply_onehot(df, category_reference):
    df = df.copy()
    for col, categories in category_reference.items():
        # Converter a coluna para tipo Categorical com as categorias fixas do train
        df[col] = pd.Categorical(df[col], categories=categories)
    # get_dummies respeita as categorias fixadas
    df_encoded = pd.get_dummies(df, columns=list(category_reference.keys()), prefix=list(category_reference.keys()))
    return df_encoded

X_train_encoded = apply_onehot(X_train, category_reference)
X_val_encoded = apply_onehot(X_val, category_reference)
X_test_encoded = apply_onehot(X_test, category_reference)

# ============================================================
# Verificação: os três conjuntos ficaram com as mesmas colunas?
# ============================================================
cols_train = set(X_train_encoded.columns)
cols_val = set(X_val_encoded.columns)
cols_test = set(X_test_encoded.columns)

print(f"\nColunas em X_train_encoded: {len(cols_train)}")
print(f"Colunas em X_val_encoded:   {len(cols_val)}")
print(f"Colunas em X_test_encoded:  {len(cols_test)}")

missing_in_val = cols_train - cols_val
missing_in_test = cols_train - cols_test
extra_in_val = cols_val - cols_train
extra_in_test = cols_test - cols_train

print(f"\nColunas em train mas não em val:  {len(missing_in_val)} (esperado: 0)")
print(f"Colunas em train mas não em test: {len(missing_in_test)} (esperado: 0)")
print(f"Colunas extra em val: {len(extra_in_val)} (esperado: 0)")
print(f"Colunas extra em test: {len(extra_in_test)} (esperado: 0)")

assert cols_train == cols_val == cols_test, "Colunas inconsistentes entre splits!"
print("\n✅ Todos os splits têm exactamente as mesmas colunas.")

Top-20 Breed1 identificadas no train: 20 raças
Cobrem 90.5% do train

Categorias por coluna (do train):
  Color1: 7 valores
  State_grouped: 4 valores
  Breed1_grouped: 21 valores

Colunas em X_train_encoded: 58
Colunas em X_val_encoded:   58
Colunas em X_test_encoded:  58

Colunas em train mas não em val:  0 (esperado: 0)
Colunas em train mas não em test: 0 (esperado: 0)
Colunas extra em val: 0 (esperado: 0)
Colunas extra em test: 0 (esperado: 0)

✅ Todos os splits têm exactamente as mesmas colunas.


In [7]:
# ============================================================
# 1. Colunas a remover do input tabular
# ============================================================
cols_to_drop = [
    'PetID',           # identificador único — overfitting garantido
    'Name',            # string livre, sinal nulo na EDA
    'Description',     # vai para o ramo de texto, não tabular
    'RescuerID',       # usado como grupo do split; se entrar como feature, destrói tudo
    'Breed2',          # secundária, baixa cobertura
    'Age',             # substituída por Age_log
    'State',           # substituída por State_grouped (já codificada)
    'Breed1',          # substituída por Breed1_grouped (já codificada) + is_mixed_breed
]

def drop_non_model_cols(df, cols_to_drop):
    df = df.copy()
    return df.drop(columns=[c for c in cols_to_drop if c in df.columns])

X_train_tab = drop_non_model_cols(X_train_encoded, cols_to_drop)
X_val_tab = drop_non_model_cols(X_val_encoded, cols_to_drop)
X_test_tab = drop_non_model_cols(X_test_encoded, cols_to_drop)

print(f"Shape após remover colunas não-modeláveis:")
print(f"  X_train_tab: {X_train_tab.shape}")
print(f"  X_val_tab:   {X_val_tab.shape}")
print(f"  X_test_tab:  {X_test_tab.shape}")

# ============================================================
# 2. Identificar colunas numéricas contínuas (a normalizar)
# ============================================================
# Binárias (0/1) e ordinais pequenas (1-4) NÃO precisam de StandardScaler.
# Só normalizamos as contínuas com ranges grandes.
numeric_cols_to_scale = ['Age_log', 'Quantity', 'Fee', 'PhotoAmt']

print(f"\nColunas a normalizar: {numeric_cols_to_scale}")
print("\nEstatísticas ANTES do scaling (no train):")
print(X_train_tab[numeric_cols_to_scale].describe().round(2))

# ============================================================
# 3. Fit do StandardScaler APENAS no train
# ============================================================
scaler = StandardScaler()
scaler.fit(X_train_tab[numeric_cols_to_scale])

# Transform aplicado aos três conjuntos (usando os parâmetros do train)
X_train_tab[numeric_cols_to_scale] = scaler.transform(X_train_tab[numeric_cols_to_scale])
X_val_tab[numeric_cols_to_scale] = scaler.transform(X_val_tab[numeric_cols_to_scale])
X_test_tab[numeric_cols_to_scale] = scaler.transform(X_test_tab[numeric_cols_to_scale])

print("\nEstatísticas DEPOIS do scaling (no train):")
print(X_train_tab[numeric_cols_to_scale].describe().round(2))

print("\nEstatísticas DEPOIS do scaling (no val):")
print(X_val_tab[numeric_cols_to_scale].describe().round(2))

# ============================================================
# 4. Converter colunas booleanas do get_dummies para int
# ============================================================
# pd.get_dummies devolve colunas bool; vários modelos preferem int (0/1)
bool_cols = X_train_tab.select_dtypes(include=['bool']).columns.tolist()
print(f"\nColunas booleanas a converter para int: {len(bool_cols)}")

for df_split in [X_train_tab, X_val_tab, X_test_tab]:
    df_split[bool_cols] = df_split[bool_cols].astype(int)

# ============================================================
# 5. Verificação final
# ============================================================
print(f"\n--- Estado final das matrizes tabulares ---")
print(f"X_train_tab: {X_train_tab.shape}, dtypes únicos: {X_train_tab.dtypes.unique()}")
print(f"X_val_tab:   {X_val_tab.shape}, dtypes únicos: {X_val_tab.dtypes.unique()}")
print(f"X_test_tab:  {X_test_tab.shape}, dtypes únicos: {X_test_tab.dtypes.unique()}")

# Confirmar zero NaNs
n_nans_train = X_train_tab.isna().sum().sum()
n_nans_val = X_val_tab.isna().sum().sum()
n_nans_test = X_test_tab.isna().sum().sum()
print(f"\nNaNs: train={n_nans_train}, val={n_nans_val}, test={n_nans_test} (esperado: 0,0,0)")

assert n_nans_train == 0 and n_nans_val == 0 and n_nans_test == 0, "Há NaNs nas matrizes!"
print("\n✅ Matrizes tabulares prontas, sem NaNs, colunas alinhadas.")

Shape após remover colunas não-modeláveis:
  X_train_tab: (9600, 50)
  X_val_tab:   (2396, 50)
  X_test_tab:  (2997, 50)

Colunas a normalizar: ['Age_log', 'Quantity', 'Fee', 'PhotoAmt']

Estatísticas ANTES do scaling (no train):
       Age_log  Quantity      Fee  PhotoAmt
count  9600.00   9600.00  9600.00   9600.00
mean      1.80      1.59    21.42      3.87
std       1.02      1.51    83.45      3.45
min       0.00      1.00     0.00      0.00
25%       1.10      1.00     0.00      2.00
50%       1.39      1.00     0.00      3.00
75%       2.56      1.00     0.00      5.00
max       5.55     20.00  3000.00     30.00

Estatísticas DEPOIS do scaling (no train):
       Age_log  Quantity      Fee  PhotoAmt
count  9600.00   9600.00  9600.00   9600.00
mean     -0.00     -0.00     0.00     -0.00
std       1.00      1.00     1.00      1.00
min      -1.76     -0.39    -0.26     -1.12
25%      -0.69     -0.39    -0.26     -0.54
50%      -0.41     -0.39    -0.26     -0.25
75%       0.75     -0.

In [8]:
# ============================================================
# 1. Criar um índice central com PetID, split, target, RescuerID
# ============================================================
# Este ficheiro é o "mapa" que liga todas as modalidades.
# Qualquer notebook que carregue texto/imagem faz join com isto via PetID.

def make_split_index(df_original, X_split, y_split, split_name):
    # X_split já não tem PetID (removemos), mas temos os índices originais
    petids = df_original.loc[X_split.index, 'PetID'].values
    rescuers = df_original.loc[X_split.index, 'RescuerID'].values
    return pd.DataFrame({
        'PetID': petids,
        'split': split_name,
        'y': y_split,
        'RescuerID': rescuers,
    })

# Nota: X_train_tab foi criado a partir de X_train (que herdou os índices de df_clean)
# Por isso X_train_tab.index aponta para as linhas originais de df_clean.
index_train = make_split_index(df_clean, X_train_tab, y_train, 'train')
index_val = make_split_index(df_clean, X_val_tab, y_val, 'val')
index_test = make_split_index(df_clean, X_test_tab, y_test, 'test')

splits_index = pd.concat([index_train, index_val, index_test], ignore_index=True)

print(f"splits_index shape: {splits_index.shape}")
print(f"Distribuição por split:")
print(splits_index['split'].value_counts())
print(f"\nPrimeiras linhas:")
print(splits_index.head())

# ============================================================
# 2. Definir os caminhos de output
# ============================================================
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(exist_ok=True)

print(f"\nDiretório de output: {PROCESSED_DIR}")

# ============================================================
# 3. Guardar splits_index (o "mapa")
# ============================================================
splits_index.to_parquet(PROCESSED_DIR / "splits_index.parquet", index=False)
print(f"✅ Guardado: splits_index.parquet ({splits_index.shape})")

# ============================================================
# 4. Guardar as matrizes tabulares
# ============================================================
# Adicionamos PetID para facilitar joins futuros; removemos antes do modelo
X_train_tab_to_save = X_train_tab.copy()
X_train_tab_to_save['PetID'] = df_clean.loc[X_train_tab.index, 'PetID'].values

X_val_tab_to_save = X_val_tab.copy()
X_val_tab_to_save['PetID'] = df_clean.loc[X_val_tab.index, 'PetID'].values

X_test_tab_to_save = X_test_tab.copy()
X_test_tab_to_save['PetID'] = df_clean.loc[X_test_tab.index, 'PetID'].values

X_train_tab_to_save.to_parquet(PROCESSED_DIR / "X_train_tab.parquet", index=False)
X_val_tab_to_save.to_parquet(PROCESSED_DIR / "X_val_tab.parquet", index=False)
X_test_tab_to_save.to_parquet(PROCESSED_DIR / "X_test_tab.parquet", index=False)

print(f"✅ Guardado: X_train_tab.parquet {X_train_tab_to_save.shape}")
print(f"✅ Guardado: X_val_tab.parquet   {X_val_tab_to_save.shape}")
print(f"✅ Guardado: X_test_tab.parquet  {X_test_tab_to_save.shape}")

# ============================================================
# 5. Guardar o scaler (para reutilizar em produção / sanity checks)
# ============================================================
import joblib
joblib.dump(scaler, PROCESSED_DIR / "standard_scaler.joblib")
print(f"✅ Guardado: standard_scaler.joblib")

# ============================================================
# 6. Guardar metadados do preprocessing (para os próximos notebooks)
# ============================================================
preprocessing_metadata = {
    'random_seed': RANDOM_SEED,
    'numeric_cols_scaled': numeric_cols_to_scale,
    'top20_breeds': top20_breeds,
    'top3_states': top_states,
    'mixed_breed_ids': sorted(MIXED_BREED_IDS),
    'category_reference': {k: list(v) for k, v in category_reference.items()},
    'cols_dropped': cols_to_drop,
    'n_features_tabular': X_train_tab.shape[1],
    'feature_columns': X_train_tab.columns.tolist(),
}

import json
with open(PROCESSED_DIR / "preprocessing_metadata.json", 'w') as f:
    json.dump(preprocessing_metadata, f, indent=2, default=str)

print(f"✅ Guardado: preprocessing_metadata.json")

# ============================================================
# 7. Verificação final — conseguimos recarregar?
# ============================================================
test_load = pd.read_parquet(PROCESSED_DIR / "X_train_tab.parquet")
print(f"\n✅ Teste de recarga: X_train_tab.parquet → shape {test_load.shape}")
print(f"Primeiras 3 colunas: {test_load.columns[:3].tolist()}")
print(f"Últimas 3 colunas:   {test_load.columns[-3:].tolist()}")

splits_index shape: (14993, 4)
Distribuição por split:
split
train    9600
test     2997
val      2396
Name: count, dtype: int64

Primeiras linhas:
       PetID  split  y                         RescuerID
0  6296e909a  train  1  3082c7125d8fb66f7dd4bff4192c8b14
1  5842f1ff5  train  1  9238e4f44c71a75282e62f7136c6b240
2  850a43f90  train  1  95481e953f8aed9ec3d16fc4509537e8
3  d24c30b4b  train  1  22fe332bf9c924d4718005891c63fbed
4  1caa6fcdb  train  1  1e0b5a458b5b77f5af581d57ebf570b3

Diretório de output: /Users/rennandamiani/Documents/ISAG/ProjetoFinal_PetFinder/data/processed
✅ Guardado: splits_index.parquet ((14993, 4))
✅ Guardado: X_train_tab.parquet (9600, 51)
✅ Guardado: X_val_tab.parquet   (2396, 51)
✅ Guardado: X_test_tab.parquet  (2997, 51)
✅ Guardado: standard_scaler.joblib
✅ Guardado: preprocessing_metadata.json

✅ Teste de recarga: X_train_tab.parquet → shape (9600, 51)
Primeiras 3 colunas: ['Type', 'Gender', 'Color2']
Últimas 3 colunas:   ['Breed1_grouped_299', 'Breed1_gr

In [9]:
# ============================================================
# 1. Verificar se sentence-transformers está instalado
# ============================================================
try:
    from sentence_transformers import SentenceTransformer
    print("✅ sentence-transformers disponível")
except ImportError:
    print("❌ sentence-transformers não instalado.")
    print("Executar: pip install sentence-transformers")
    raise

# ============================================================
# 2. Preparar as descrições — substituir NaN e textos muito curtos por ""
# ============================================================
# Regra do CLAUDE.md: descrições com menos de 5 palavras são tratadas como vazias
# Isso afeta ~5.4% dos animais

def clean_description(text):
    if pd.isna(text):
        return ""
    text = str(text).strip()
    if len(text.split()) < 5:
        return ""
    return text

# Aplicar ao df_clean para termos um dicionário PetID → descrição limpa
descriptions_df = pd.DataFrame({
    'PetID': df_clean['PetID'].values,
    'description_raw': df_clean['Description'].values,
})
descriptions_df['description_clean'] = descriptions_df['description_raw'].apply(clean_description)

# ============================================================
# 3. Estatísticas — quantas descrições vazias temos?
# ============================================================
n_empty = (descriptions_df['description_clean'] == "").sum()
n_nan = descriptions_df['description_raw'].isna().sum()
n_short = ((descriptions_df['description_raw'].notna()) & 
           (descriptions_df['description_clean'] == "")).sum()

print(f"\nTotal de descrições: {len(descriptions_df)}")
print(f"  NaN originais:          {n_nan} ({n_nan/len(descriptions_df)*100:.2f}%)")
print(f"  Curtas (<5 palavras):   {n_short} ({n_short/len(descriptions_df)*100:.2f}%)")
print(f"  Vazias no total:        {n_empty} ({n_empty/len(descriptions_df)*100:.2f}%)")
print(f"  Válidas:                {len(descriptions_df) - n_empty}")

# Distribuição do comprimento das descrições válidas
valid_descs = descriptions_df[descriptions_df['description_clean'] != ""]
word_counts = valid_descs['description_clean'].apply(lambda x: len(x.split()))
print(f"\nComprimento das descrições válidas (em palavras):")
print(f"  Mediana: {word_counts.median():.0f}")
print(f"  P95:     {word_counts.quantile(0.95):.0f}")
print(f"  Máximo:  {word_counts.max():.0f}")
print(f"  (O MiniLM trunca a 256 tokens — seguro)")

# ============================================================
# 4. Alinhar a ordem com splits_index para não haver confusão depois
# ============================================================
# splits_index.PetID define a ORDEM canónica. Garantimos que o vetor de
# embeddings vai estar na mesma ordem.
descriptions_aligned = splits_index.merge(
    descriptions_df[['PetID', 'description_clean']],
    on='PetID',
    how='left'
)

# Verificar que ninguém se perdeu no merge
assert len(descriptions_aligned) == len(splits_index), "Desalinhamento no merge!"
assert descriptions_aligned['description_clean'].notna().all() or \
       (descriptions_aligned['description_clean'] == "").any(), "Há NaNs inesperados!"

print(f"\n✅ Descrições alinhadas com splits_index: {descriptions_aligned.shape}")
print(f"Primeiras linhas:")
print(descriptions_aligned.head(3)[['PetID', 'split', 'description_clean']])

✅ sentence-transformers disponível

Total de descrições: 14993
  NaN originais:          13 (0.09%)
  Curtas (<5 palavras):   796 (5.31%)
  Vazias no total:        809 (5.40%)
  Válidas:                14184

Comprimento das descrições válidas (em palavras):
  Mediana: 47
  P95:     186
  Máximo:  1257
  (O MiniLM trunca a 256 tokens — seguro)

✅ Descrições alinhadas com splits_index: (14993, 5)
Primeiras linhas:
       PetID  split                                  description_clean
0  6296e909a  train  I just found it alone yesterday near my apartm...
1  5842f1ff5  train  Good guard dog, very alert, active, obedience ...
2  850a43f90  train  This handsome yet cute boy is up for adoption....


In [10]:
import numpy as np
from sentence_transformers import SentenceTransformer
import time

# ============================================================
# 1. Carregar o modelo MiniLM
# ============================================================
# all-MiniLM-L6-v2: 22M parâmetros, 384 dimensões de output
# Na primeira execução descarrega ~90MB para ~/.cache/huggingface/
print("A carregar modelo MiniLM...")
start = time.time()
model_text = SentenceTransformer('all-MiniLM-L6-v2')
print(f"Modelo carregado em {time.time()-start:.1f}s")
print(f"Dimensão do embedding: {model_text.get_sentence_embedding_dimension()}")
print(f"Comprimento máximo de tokens: {model_text.max_seq_length}")

# ============================================================
# 2. Computar embeddings (na ordem do splits_index)
# ============================================================
texts_to_embed = descriptions_aligned['description_clean'].tolist()

print(f"\nA gerar embeddings para {len(texts_to_embed)} descrições...")
print("(Descrições vazias produzem embeddings válidos mas consistentes — o modelo lida bem)")

start = time.time()
text_embeddings = model_text.encode(
    texts_to_embed,
    batch_size=64,              # lotes para aproveitar a GPU/CPU
    show_progress_bar=True,     # barra de progresso
    convert_to_numpy=True,      # devolve numpy array (não tensor)
    normalize_embeddings=False, # normalização fica para decidir mais tarde
)
elapsed = time.time() - start
print(f"\n✅ Embeddings gerados em {elapsed:.1f}s ({len(texts_to_embed)/elapsed:.0f} descrições/s)")
print(f"Shape final: {text_embeddings.shape}")
print(f"dtype: {text_embeddings.dtype}")

# ============================================================
# 3. Verificação — embeddings de descrições vazias vs cheias
# ============================================================
# Sanity check: embeddings de "" devem ser todos iguais entre si,
# mas diferentes de uma descrição com conteúdo.
empty_mask = (descriptions_aligned['description_clean'] == "").values

empty_embeddings = text_embeddings[empty_mask]
valid_embeddings = text_embeddings[~empty_mask]

print(f"\nEmbeddings de descrições vazias: {empty_embeddings.shape}")
print(f"Embeddings de descrições válidas: {valid_embeddings.shape}")

# Todos os embeddings de "" devem ser idênticos
if empty_embeddings.shape[0] > 1:
    max_diff_empty = np.abs(empty_embeddings[0] - empty_embeddings[1]).max()
    print(f"Diferença máxima entre dois embeddings de '': {max_diff_empty:.6f} (esperado: 0)")

# Norma média — só para termos noção dos valores
print(f"\nNorma média embeddings válidos: {np.linalg.norm(valid_embeddings, axis=1).mean():.3f}")
print(f"Norma média embeddings vazios:  {np.linalg.norm(empty_embeddings, axis=1).mean():.3f}")

# ============================================================
# 4. Guardar em disco
# ============================================================
# Guardamos DOIS ficheiros: os embeddings e os PetIDs (por ordem correspondente).
# O PetID é a chave para ligar ao splits_index.

petids_text = descriptions_aligned['PetID'].values

np.save(CACHE_DIR / "text_embeddings_minilm.npy", text_embeddings)
np.save(CACHE_DIR / "text_embeddings_petids.npy", petids_text)

print(f"\n✅ Guardado: text_embeddings_minilm.npy  (shape {text_embeddings.shape}, "
      f"{text_embeddings.nbytes / 1e6:.1f}MB)")
print(f"✅ Guardado: text_embeddings_petids.npy  (shape {petids_text.shape})")

# ============================================================
# 5. Teste de recarga — podemos ler o que acabámos de escrever?
# ============================================================
reload_embeddings = np.load(CACHE_DIR / "text_embeddings_minilm.npy")
reload_petids = np.load(CACHE_DIR / "text_embeddings_petids.npy", allow_pickle=True)

assert reload_embeddings.shape == text_embeddings.shape
assert np.array_equal(reload_petids, petids_text)
print(f"\n✅ Teste de recarga: embeddings e PetIDs recuperados sem perda.")

A carregar modelo MiniLM...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Modelo carregado em 3.6s
Dimensão do embedding: 384
Comprimento máximo de tokens: 256

A gerar embeddings para 14993 descrições...
(Descrições vazias produzem embeddings válidos mas consistentes — o modelo lida bem)


/var/folders/jy/4lj4gvfs15j22x8x1xygrbfm0000gn/T/ipykernel_77646/2459196646.py:14: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Dimensão do embedding: {model_text.get_sentence_embedding_dimension()}")


Batches:   0%|          | 0/235 [00:00<?, ?it/s]


✅ Embeddings gerados em 22.8s (657 descrições/s)
Shape final: (14993, 384)
dtype: float32

Embeddings de descrições vazias: (809, 384)
Embeddings de descrições válidas: (14184, 384)
Diferença máxima entre dois embeddings de '': 0.000000 (esperado: 0)

Norma média embeddings válidos: 1.000
Norma média embeddings vazios:  1.000

✅ Guardado: text_embeddings_minilm.npy  (shape (14993, 384), 23.0MB)
✅ Guardado: text_embeddings_petids.npy  (shape (14993,))

✅ Teste de recarga: embeddings e PetIDs recuperados sem perda.


In [11]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing import image as keras_image
from PIL import Image
import warnings

# ============================================================
# 1. Verificar GPU disponível
# ============================================================
gpus = tf.config.list_physical_devices('GPU')
print(f"TensorFlow version: {tf.__version__}")
print(f"GPUs disponíveis: {len(gpus)}")
for gpu in gpus:
    print(f"  - {gpu}")

# ============================================================
# 2. Carregar MobileNetV2 sem a camada final de classificação
# ============================================================
# include_top=False → remove a camada de classificação (1000 classes ImageNet)
# pooling='avg' → adiciona global average pooling, devolve vetor 1280-dim
print("\nA carregar MobileNetV2 (pré-treinado no ImageNet)...")
start = time.time()
model_img = MobileNetV2(
    weights='imagenet',
    include_top=False,
    pooling='avg',
    input_shape=(224, 224, 3),
)
print(f"Modelo carregado em {time.time()-start:.1f}s")
print(f"Dimensão do embedding: {model_img.output_shape[-1]}")
print(f"Input shape esperado: {model_img.input_shape}")

# ============================================================
# 3. Identificar imagens existentes
# ============================================================
# Para cada PetID, verificamos se PetID-1.jpg existe no disco
IMAGES_DIR = DATA_DIR / "train_images"

def get_image_path(pet_id):
    path = IMAGES_DIR / f"{pet_id}-1.jpg"
    return path if path.exists() else None

# Aplicamos ao splits_index para mantermos a ORDEM canónica
image_paths_df = splits_index[['PetID']].copy()
image_paths_df['image_path'] = image_paths_df['PetID'].apply(get_image_path)
image_paths_df['has_image'] = image_paths_df['image_path'].notna()

n_with = image_paths_df['has_image'].sum()
n_without = (~image_paths_df['has_image']).sum()

print(f"\nResumo das imagens:")
print(f"  Com PetID-1.jpg no disco: {n_with} ({n_with/len(image_paths_df)*100:.2f}%)")
print(f"  Sem imagem:               {n_without} ({n_without/len(image_paths_df)*100:.2f}%)")
print(f"  (esperado do NB01: ~14652 com foto, ~341 sem foto)")

# ============================================================
# 4. Testar o pipeline numa única imagem (sanity check rápido)
# ============================================================
# Antes de processar 15k imagens, garantimos que uma funciona.
sample_path = image_paths_df.loc[image_paths_df['has_image'], 'image_path'].iloc[0]
print(f"\nTeste numa imagem exemplo: {sample_path.name}")

def load_and_preprocess_image(path):
    """Carrega imagem, redimensiona para 224x224, aplica preprocess do MobileNetV2."""
    img = Image.open(path).convert('RGB')       # garante 3 canais (alguns JPEG têm 4 ou 1)
    img = img.resize((224, 224), Image.BILINEAR)
    arr = np.array(img, dtype=np.float32)        # shape (224, 224, 3), valores 0-255
    arr = preprocess_input(arr)                  # normaliza para [-1, 1] (padrão MobileNetV2)
    return arr

sample_arr = load_and_preprocess_image(sample_path)
print(f"  Shape da imagem processada: {sample_arr.shape}")
print(f"  Range de valores: [{sample_arr.min():.2f}, {sample_arr.max():.2f}] (esperado: [-1, 1])")

# Correr através do modelo (com batch dummy de 1)
sample_batch = np.expand_dims(sample_arr, axis=0)  # shape (1, 224, 224, 3)
sample_embedding = model_img.predict(sample_batch, verbose=0)
print(f"  Embedding shape: {sample_embedding.shape}")
print(f"  Embedding norm:  {np.linalg.norm(sample_embedding):.2f}")

print("\n✅ Pipeline de imagem testado com sucesso. Pronto para processar todas as imagens.")

TensorFlow version: 2.16.2
GPUs disponíveis: 1
  - PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')

A carregar MobileNetV2 (pré-treinado no ImageNet)...


2026-04-20 20:17:59.777347: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3 Pro
2026-04-20 20:17:59.777391: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 36.00 GB
2026-04-20 20:17:59.777394: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 14.04 GB
2026-04-20 20:17:59.777422: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-04-20 20:17:59.777433: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Modelo carregado em 1.3s
Dimensão do embedding: 1280
Input shape esperado: (None, 224, 224, 3)

Resumo das imagens:
  Com PetID-1.jpg no disco: 14652 (97.73%)
  Sem imagem:               341 (2.27%)
  (esperado do NB01: ~14652 com foto, ~341 sem foto)

Teste numa imagem exemplo: 6296e909a-1.jpg
  Shape da imagem processada: (224, 224, 3)
  Range de valores: [-0.98, 1.00] (esperado: [-1, 1])


2026-04-20 20:18:01.798345: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


  Embedding shape: (1, 1280)
  Embedding norm:  21.12

✅ Pipeline de imagem testado com sucesso. Pronto para processar todas as imagens.


In [12]:
from tqdm import tqdm

# ============================================================
# 1. Configuração do batch
# ============================================================
BATCH_SIZE = 64  # Número de imagens processadas em paralelo pela GPU
EMBEDDING_DIM = 1280
N_TOTAL = len(image_paths_df)

# Pré-alocar o array de saída — mais eficiente que fazer append
image_embeddings = np.zeros((N_TOTAL, EMBEDDING_DIM), dtype=np.float32)

print(f"Total de animais: {N_TOTAL}")
print(f"Com imagem: {image_paths_df['has_image'].sum()}")
print(f"Sem imagem (zeros): {(~image_paths_df['has_image']).sum()}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Número de batches: {int(np.ceil(N_TOTAL / BATCH_SIZE))}")

# ============================================================
# 2. Função que processa um batch de paths
# ============================================================
def process_batch(paths_batch):
    """
    Recebe uma lista de paths (ou None para sem imagem).
    Devolve um array (batch_size, 1280) de embeddings.
    """
    # Carregar e pré-processar cada imagem do batch
    batch_arrays = []
    for path in paths_batch:
        if path is None:
            # Placeholder de zeros para animais sem foto
            # (normalizado: zeros antes do preprocess continuam zeros na escala [-1, 1])
            arr = np.zeros((224, 224, 3), dtype=np.float32)
        else:
            try:
                arr = load_and_preprocess_image(path)
            except Exception as e:
                # Fallback: se uma imagem específica der erro, usar zeros
                print(f"⚠️  Erro em {path.name}: {e} — a usar zeros")
                arr = np.zeros((224, 224, 3), dtype=np.float32)
        batch_arrays.append(arr)
    
    # Empilhar em batch (batch_size, 224, 224, 3)
    batch_tensor = np.stack(batch_arrays, axis=0)
    
    # Forward pass no modelo
    embeddings = model_img.predict(batch_tensor, verbose=0)
    return embeddings

# ============================================================
# 3. Loop principal — processar todos os batches
# ============================================================
paths_list = image_paths_df['image_path'].tolist()

print(f"\nA processar {N_TOTAL} imagens em batches de {BATCH_SIZE}...")
start = time.time()

# Suprimir warnings irritantes do TF durante o loop
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    
    for batch_start in tqdm(range(0, N_TOTAL, BATCH_SIZE), desc="Batches"):
        batch_end = min(batch_start + BATCH_SIZE, N_TOTAL)
        paths_batch = paths_list[batch_start:batch_end]
        
        batch_embeddings = process_batch(paths_batch)
        image_embeddings[batch_start:batch_end] = batch_embeddings

elapsed = time.time() - start
print(f"\n✅ Embeddings gerados em {elapsed:.1f}s ({N_TOTAL/elapsed:.1f} imgs/s)")
print(f"Shape final: {image_embeddings.shape}")
print(f"dtype: {image_embeddings.dtype}")
print(f"Tamanho em memória: {image_embeddings.nbytes / 1e6:.1f} MB")

# ============================================================
# 4. Sanity checks
# ============================================================
# Separar embeddings de imagens reais vs placeholders
has_image_mask = image_paths_df['has_image'].values
real_embeddings = image_embeddings[has_image_mask]
zero_embeddings = image_embeddings[~has_image_mask]

print(f"\n--- Verificação de qualidade ---")
print(f"Embeddings de imagens reais: {real_embeddings.shape}")
print(f"Embeddings de placeholders (sem foto): {zero_embeddings.shape}")

# Norma média — imagens reais devem ter norma >>0; placeholders devem ter norma == 0 ou próxima
real_norms = np.linalg.norm(real_embeddings, axis=1)
zero_norms = np.linalg.norm(zero_embeddings, axis=1)

print(f"\nNorma dos embeddings reais:")
print(f"  Mediana: {np.median(real_norms):.2f}")
print(f"  Min:     {real_norms.min():.2f}")
print(f"  Max:     {real_norms.max():.2f}")

print(f"\nNorma dos embeddings placeholder (esperado: tudo igual, próximo de 0 ou valor constante):")
print(f"  Mediana: {np.median(zero_norms):.4f}")
print(f"  Min:     {zero_norms.min():.4f}")
print(f"  Max:     {zero_norms.max():.4f}")

# Verificar que placeholders são todos idênticos entre si
if len(zero_embeddings) > 1:
    max_diff = np.abs(zero_embeddings[0] - zero_embeddings[1]).max()
    print(f"\nDiferença máxima entre dois placeholders: {max_diff:.6f} (esperado: 0)")

Total de animais: 14993
Com imagem: 14652
Sem imagem (zeros): 341
Batch size: 64
Número de batches: 235

A processar 14993 imagens em batches de 64...


Batches: 100%|██████████| 235/235 [01:01<00:00,  3.81it/s]


✅ Embeddings gerados em 61.7s (242.9 imgs/s)
Shape final: (14993, 1280)
dtype: float32
Tamanho em memória: 76.8 MB

--- Verificação de qualidade ---
Embeddings de imagens reais: (14652, 1280)
Embeddings de placeholders (sem foto): (341, 1280)

Norma dos embeddings reais:
  Mediana: 23.48
  Min:     15.00
  Max:     42.50

Norma dos embeddings placeholder (esperado: tudo igual, próximo de 0 ou valor constante):
  Mediana: 24.6350
  Min:     24.6350
  Max:     24.6350

Diferença máxima entre dois placeholders: 0.000000 (esperado: 0)


In [13]:
# ============================================================
# 1. Extrair PetIDs na ordem correcta
# ============================================================
# image_paths_df tem a mesma ordem que splits_index (construído a partir dele)
petids_image = image_paths_df['PetID'].values

# Verificação: ordem dos PetIDs deve bater com o splits_index
assert np.array_equal(petids_image, splits_index['PetID'].values), \
    "Desalinhamento entre image_paths_df e splits_index!"
print("✅ Ordem dos PetIDs alinhada com splits_index")

# ============================================================
# 2. Guardar em disco
# ============================================================
np.save(CACHE_DIR / "image_embeddings_mobilenetv2.npy", image_embeddings)
np.save(CACHE_DIR / "image_embeddings_petids.npy", petids_image)

print(f"\n✅ Guardado: image_embeddings_mobilenetv2.npy "
      f"(shape {image_embeddings.shape}, {image_embeddings.nbytes / 1e6:.1f}MB)")
print(f"✅ Guardado: image_embeddings_petids.npy (shape {petids_image.shape})")

# ============================================================
# 3. Teste de recarga
# ============================================================
reload_embeddings = np.load(CACHE_DIR / "image_embeddings_mobilenetv2.npy")
reload_petids = np.load(CACHE_DIR / "image_embeddings_petids.npy", allow_pickle=True)

assert reload_embeddings.shape == image_embeddings.shape
assert np.array_equal(reload_petids, petids_image)
assert np.allclose(reload_embeddings, image_embeddings), "Embeddings mudaram após recarga!"
print(f"\n✅ Teste de recarga: embeddings e PetIDs recuperados sem perda.")

# ============================================================
# 4. Resumo final do NB02
# ============================================================
print("\n" + "="*60)
print("🎉 NB02 COMPLETO — Resumo dos artefactos produzidos")
print("="*60)

print(f"\n📁 data/processed/")
for f in sorted((PROCESSED_DIR).glob("*")):
    size_mb = f.stat().st_size / 1e6
    print(f"   {f.name:40s} {size_mb:>6.2f} MB")

print(f"\n📁 data/cache/")
for f in sorted((CACHE_DIR).glob("*")):
    size_mb = f.stat().st_size / 1e6
    print(f"   {f.name:40s} {size_mb:>6.2f} MB")

print("\n✅ Pipeline pronto para os baselines (NB03, NB04, NB05) e multimodal (NB06).")

✅ Ordem dos PetIDs alinhada com splits_index

✅ Guardado: image_embeddings_mobilenetv2.npy (shape (14993, 1280), 76.8MB)
✅ Guardado: image_embeddings_petids.npy (shape (14993,))

✅ Teste de recarga: embeddings e PetIDs recuperados sem perda.

🎉 NB02 COMPLETO — Resumo dos artefactos produzidos

📁 data/processed/
   X_test_tab.parquet                         0.09 MB
   X_train_tab.parquet                        0.22 MB
   X_val_tab.parquet                          0.08 MB
   preprocessing_metadata.json                0.00 MB
   splits_index.parquet                       0.39 MB
   standard_scaler.joblib                     0.00 MB

📁 data/cache/
   image_embeddings_mobilenetv2.npy          76.76 MB
   image_embeddings_petids.npy                0.28 MB
   text_embeddings_minilm.npy                23.03 MB
   text_embeddings_petids.npy                 0.28 MB

✅ Pipeline pronto para os baselines (NB03, NB04, NB05) e multimodal (NB06).
